# 03 · Validation and guardrails

**Role of this layer.** "The LLM proposes, the system disposes." A plan is inert
JSON; the validator decides, *without touching any data values*, whether it may run:

* **BLOCK**: cannot run (unknown capability, bad parameter, returns of a yield...).
* **APPROVAL**: runs only after a human explicitly approves (look-ahead, material assumptions).
* **WARN**: runs; the warning is shown and passed to the narrator.

It does a metadata *dry run*: walking the steps and propagating data type, kind
and frequency through the registry, which is how it catches "returns of a yield"
or "the spread of a daily and a weekly series" before anything executes.

Below, every static guardrail G01-G13 is triggered on a deliberately broken plan
(G11 was retired together with the statistical tests; codes are never reused).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
import copy

from quantlab.data import load_catalog
from quantlab.plan import Plan
from quantlab.scenarios import SCENARIOS_BY_ID
from quantlab.validation import GUARDRAILS, check_schema, fill_defaults, validate_plan

catalog = load_catalog(settings.catalog_path)


def scenario_plan(scenario_id):
    """A mutable copy of a demo scenario's canned plan."""
    s = SCENARIOS_BY_ID[scenario_id]
    return {"plan_id": "demo", "request": s.request, **copy.deepcopy(s.plan)}


def codes_for(raw, **budgets):
    """Parse and validate a plan dict; return the issue codes raised."""
    plan, schema_issues = check_schema(raw)
    if plan is None:
        return sorted({i.code for i in schema_issues})
    return validate_plan(fill_defaults(plan), catalog, **budgets).codes()


pd.DataFrame(
    [
        {
            "code": g.code,
            "name": g.name,
            "severity": g.severity.value,
            "stage": g.stage,
            "what goes wrong without it": g.risk,
        }
        for g in GUARDRAILS.values()
        if g.stage == "static"
    ]
)

,code,name,severity,stage,what goes wrong without it
0,G01,SCHEMA_INVALID,BLOCK,static,"Malformed or partial planner output would have to be guessed at, and extra fields (e.g. 'code') could slip..."
1,G02,UNKNOWN_CAPABILITY,BLOCK,static,"The LLM could ask for an analysis nobody wrote or tested (e.g. a forecasting model, or 'run Python')."
2,G03,INVALID_PARAM,BLOCK,static,"Typos, wrong types or extreme values (window=1, quantile=1.5) would crash or silently produce nonsense."
3,G04,BAD_REFERENCE,BLOCK,static,"Steps could read missing outputs or outputs of later steps, leaving the order of computation undefined."
4,G05,TYPE_MISMATCH,BLOCK,static,"Meaningless chains would run, e.g. volatility of a price level or summary statistics of regime flags."
5,G06,BUDGET_EXCEEDED,BLOCK,static,Runaway plans could use unbounded compute and become impossible for a human to review.
6,G07,UNKNOWN_DATASET,BLOCK,static,The LLM could invent a dataset id or reach for data outside the approved catalog.
7,G08,RETURNS_ON_RATES,BLOCK,static,'Returns' of a yield are meaningless (1% -> 2% is not +100%); rates must be differenced.
8,G09,FREQUENCY_MISMATCH,BLOCK,static,Combining a daily with a weekly series silently keeps only the dates they share and mixes frequencies.
9,G10,LOOK_AHEAD,APPROVAL,static,"Full-sample thresholds use future data, so results look better than anything achievable in real time."


A clean plan (scenario E) raises nothing; scenario A only needs the return-method approval.

In [3]:
print("E:", codes_for(scenario_plan("E")))
print("A:", codes_for(scenario_plan("A")))

E: []
A: ['G12']


Now one crafted plan per guardrail:

In [4]:
cases = {}

raw = scenario_plan("A")
raw["steps"][0]["code"] = "import os"
cases["G01 extra field 'code'"] = raw

cases["G02 prediction: no forecast capability (scenario C)"] = scenario_plan("C")

raw = scenario_plan("B")
raw["steps"][2]["params"] = {"window": 2}
cases["G03 window below range"] = raw

raw = scenario_plan("B")
raw["steps"][2]["inputs"] = {"series": "s4"}
cases["G04 reference to a later step"] = raw

raw = scenario_plan("B")
raw["steps"][2]["inputs"] = {"series": "s1"}
cases["G05 volatility of a price level"] = raw

raw = scenario_plan("E")
raw["steps"] += [
    {"id": f"s{i}", "capability": "summary_stats", "inputs": {"series": "s3"}} for i in range(5, 17)
]
cases["G06 16 steps > budget of 15"] = raw

raw = scenario_plan("E")
raw["steps"][1]["params"] = {"dataset_id": "BITCOIN"}
cases["G07 dataset not in catalog"] = raw

raw = scenario_plan("E")
raw["steps"] = [
    raw["steps"][0],
    {
        "id": "s2",
        "capability": "compute_returns",
        "inputs": {"series": "s1"},
        "params": {"method": "simple"},
    },
]
cases["G08 returns of a yield"] = raw

raw = scenario_plan("E")
raw["steps"].insert(
    2, {"id": "s1w", "capability": "resample", "inputs": {"series": "s1"}, "params": {"freq": "W"}}
)
raw["steps"][3]["inputs"] = {"a": "s1w", "b": "s2"}
cases["G09 spread of a weekly and a daily yield"] = raw

raw = scenario_plan("B")
raw["steps"][3]["params"]["method"] = "full_sample"
cases["G10 full-sample threshold"] = raw

cases["G12 return method needs approval (scenario A)"] = scenario_plan("A")
cases["G13 planner refusal (scenario D)"] = scenario_plan("D")

pd.DataFrame(
    [{"case": name, "codes raised": ", ".join(codes_for(raw))} for name, raw in cases.items()]
)

,case,codes raised
0,G01 extra field 'code',G01
1,G02 prediction: no forecast capability (scenario C),G02
2,G03 window below range,"G03, G12"
3,G04 reference to a later step,"G04, G12"
4,G05 volatility of a price level,"G05, G12"
5,G06 16 steps > budget of 15,G06
6,G07 dataset not in catalog,G07
7,G08 returns of a yield,"G08, G12"
8,G09 spread of a weekly and a daily yield,G09
9,G10 full-sample threshold,"G10, G12"


Each issue carries a message and a suggestion. For example, scenario C (a
prediction, for which no capability exists):

In [5]:
report = validate_plan(fill_defaults(Plan(**scenario_plan("C"))), catalog)
for issue in report.issues:
    print(f"[{issue.severity}] {issue.issue_id}: {issue.message}\n    -> {issue.suggestion}")

[BLOCK] G02:s2: 'forecast' is not a registered capability.
    -> Use only: load_series, resample, compute_returns, difference, spread, rolling_volatility, flag_regime, summary_stats. New capabilities are added by developers, with tests, not by the planner.


The dry run: predicted output metadata of each step of scenario B (no data touched).

In [6]:
report = validate_plan(fill_defaults(Plan(**scenario_plan("B"))), catalog)
pd.DataFrame(report.dry_run).T[["data_type", "kind", "frequency", "return_method", "dataset_ids"]]

,data_type,kind,frequency,return_method,dataset_ids
s1,series,price_level,D,None,[NASDAQCOM]
s2,series,returns,D,simple,[NASDAQCOM]
s3,series,volatility,D,simple,[NASDAQCOM]
s4,flags,flags,D,simple,[NASDAQCOM]
